# 01 — Per-Class Analysis

Deep dive into individual class performance. Identify which classes are easy/hard,
and explore how performance varies with observation count and redshift.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../.."))
from config import DATA_CONFIG
from src.metrics import per_class_metrics
from src.utils import get_class_name, save_figure

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

In [ ]:
# Load the best model's predictions (update this to whichever performed better)
best_data = np.load('../../data/rnn_test_preds.npz', allow_pickle=True)
y_true = best_data['y_true']
y_pred = best_data['y_pred']
y_proba = best_data['y_pred_proba']
class_names = list(best_data['class_names'])

metadata = pd.read_parquet(os.path.join(DATA_CONFIG['processed_dir'], 'training_metadata.parquet'))
lc = pd.read_parquet(os.path.join(DATA_CONFIG['processed_dir'], 'training_lc.parquet'))
print(f"Test set: {len(y_true)} objects")

## 1. Most-Confused Class Pairs

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)
n_classes = len(class_names)

# Off-diagonal elements sorted by count
confusions = []
for i in range(n_classes):
    for j in range(n_classes):
        if i != j and cm[i, j] > 0:
            confusions.append({
                'true': class_names[i],
                'predicted': class_names[j],
                'count': cm[i, j],
                'pct_of_true': 100 * cm[i, j] / cm[i].sum() if cm[i].sum() > 0 else 0,
            })

conf_df = pd.DataFrame(confusions).sort_values('count', ascending=False)
print("Top 15 confusions:")
print(conf_df.head(15).to_string(index=False))

## 2. Galactic vs Extragalactic Performance

In [ ]:
from src.utils import decode_labels, encode_labels

# Reconstruct original targets for the test set
# We need the decode_map
all_targets = metadata['target'].values
_, encode_map, decode_map = encode_labels(all_targets)

test_original = np.array([decode_map[y] for y in y_true])

galactic_mask = np.isin(test_original, list(DATA_CONFIG['galactic_classes']))
extragalactic_mask = ~galactic_mask

gal_acc = np.mean(y_true[galactic_mask] == y_pred[galactic_mask])
ext_acc = np.mean(y_true[extragalactic_mask] == y_pred[extragalactic_mask])
print(f"Galactic accuracy:       {gal_acc:.4f} (n={galactic_mask.sum()})")
print(f"Extragalactic accuracy:  {ext_acc:.4f} (n={extragalactic_mask.sum()})")

## 3. Performance vs Number of Observations

In [ ]:
# Count observations per object in the test set
# Note: we need to match object IDs to test set indices
# This requires knowing which objects are in the test set
obs_counts = lc.groupby('object_id').size().rename('n_obs')

# Bin by observation count and compute accuracy per bin
# Using all test objects
all_obs = metadata.merge(obs_counts, left_on='object_id', right_index=True)

# Create bins
bins = [0, 50, 100, 150, 200, 300, 500, 1000]
print("Overall observation count distribution:")
print(pd.cut(all_obs['n_obs'], bins=bins).value_counts().sort_index())

## 4. Rare Class Deep Dive

Kilonova (64) and SLSN (95) are the rarest classes — how do we perform on them?

In [ ]:
rare_classes = [64, 95, 15, 6]  # KN, SLSN, TDE, Microlensing

for cls_id in rare_classes:
    cls_name = get_class_name(cls_id)
    if cls_id in encode_map:
        encoded = encode_map[cls_id]
        mask = y_true == encoded
        n_test = mask.sum()
        if n_test > 0:
            correct = (y_pred[mask] == encoded).sum()
            print(f"{cls_name} ({cls_id}): {correct}/{n_test} correct ({100*correct/n_test:.1f}%)")
            # What do we misclassify them as?
            wrong_mask = mask & (y_pred != encoded)
            if wrong_mask.sum() > 0:
                wrong_preds = y_pred[wrong_mask]
                for pred_cls, count in zip(*np.unique(wrong_preds, return_counts=True)):
                    print(f"  -> misclassified as {class_names[pred_cls]}: {count}")
            print()

In [ ]:
# Per-class metrics summary
metrics_df = per_class_metrics(y_true, y_pred,
                               {i: class_names[i] for i in range(len(class_names))})

fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(metrics_df))
ax.bar(x, metrics_df['f1'], color='#0072B2', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(metrics_df['class'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 Score (Best Model)')
ax.set_ylim(0, 1.05)

# Annotate support
for i, (f1, sup) in enumerate(zip(metrics_df['f1'], metrics_df['support'])):
    ax.text(i, f1 + 0.02, f"n={sup}", ha='center', fontsize=7, color='gray')

plt.tight_layout()
save_figure(fig, 'per_class_analysis')
plt.show()